In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to system path for importing src modules
sys.path.append('../')
from src.processing import load_data

# Set visual style for plots
sns.set_theme(style="whitegrid")
%matplotlib inline


In [ ]:
# Load the credit card dataset using our modular load_data function
try:
    credit_df = load_data('../data/raw/creditcard.csv')
except FileNotFoundError:
    print("Please make sure 'creditcard.csv' is in your 'data/raw/' directory.")

In [ ]:
print("--- Initial Data Assessment ---")
print(f"Dataset Shape: {credit_df.shape}")

# 1. Check for Missing Values
missing_values = credit_df.isnull().sum().sum()
print(f"Total Missing Values: {missing_values}")

# 2. Check and Remove Duplicates
duplicate_count = credit_df.duplicated().sum()
print(f"Number of Duplicate Rows Identified: {duplicate_count}")

if duplicate_count > 0:
    credit_df = credit_df.drop_duplicates()
    print(f"Duplicates successfully removed. New shape: {credit_df.shape}")
else:
    print("No duplicates detected.")

# 3. Correct Data Types
# The features V1-V28, Time, Amount, and Class are typically loaded with correct numeric datatypes.
# Let's verify to ensure clean typing.
print("\n--- Data Types Check ---")
print(credit_df.dtypes.value_counts())

In [ ]:
# Calculate absolute and relative class distributions
class_counts = credit_df['Class'].value_counts()
class_pct = credit_df['Class'].value_counts(normalize=True) * 100

print("--- Class Distribution (Class Imbalance) ---")
print(f"Legitimate Transactions (Class 0): {class_counts[0]} ({class_pct[0]:.4f}%)")
print(f"Fraudulent Transactions (Class 1): {class_counts[1]} ({class_pct[1]:.4f}%)")

# Visualize Class Imbalance
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=credit_df, palette='Set2')
plt.title('Credit Card Transaction Class Imbalance (Log Scale)')
plt.yscale('log') # Log scale because legitimate heavily dwarfs fraud
plt.ylabel('Count (Log Scale)')
plt.xlabel('Class (0 = Legitimate, 1 = Fraud)')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Distribution of Transaction Amount
sns.histplot(credit_df['Amount'], bins=50, kde=True, ax=ax[0], color='purple')
ax[0].set_title('Distribution of Transaction Amount')
ax[0].set_yscale('log') # Log scale since most amounts are small but outliers exist
ax[0].set_ylabel('Density (Log Scale)')

# Distribution of Transaction Time
sns.histplot(credit_df['Time'], bins=50, kde=True, ax=ax[1], color='teal')
ax[1].set_title('Distribution of Transaction Time (Seconds)')
ax[1].set_xlabel('Time (seconds since first transaction)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Bivariate Relationship: Transaction Amount by Class (Log Scale boxplot)
sns.boxplot(x='Class', y='Amount', data=credit_df, ax=ax[0], palette='Set2')
ax[0].set_title('Transaction Amount distribution by Class')
ax[0].set_yscale('log')
ax[0].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])

# Bivariate Relationship: Transaction Time by Class
sns.kdeplot(data=credit_df, x='Time', hue='Class', common_norm=False, fill=True, ax=ax[1], palette='Set2')
ax[1].set_title('Transaction Density over Time by Class')
ax[1].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

In [ ]:
# Ensure data/processed directory exists
os.makedirs('../data/processed', exist_ok=True)

# Save the cleaned dataset for modeling and feature engineering
credit_df.to_csv('../data/processed/cleaned_creditcard.csv', index=False)
print("Cleaned credit card dataset successfully saved to '../data/processed/cleaned_creditcard.csv'")